In [1]:
# ============================================================
# AWS GLUE E-COMMERCE ETL
# Historical CSV + Ordering API JSON
# S3 Curated + Amazon Redshift
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from functools import reduce

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: 5afd4507-4986-4fe6-81c9-a16bf6b537f9
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 5afd4507-4986-4fe6-81c9-a16bf6b537f9 to get into ready status...
Session 5afd4507-4986-4fe6-81c9-a16bf6b537f9 has been created.



In [80]:
# ============================================================
# 1. SPARK SESSION
# ============================================================

spark = SparkSession.builder \
    .appName("Ecommerce-Historical-API-ETL") \
    .getOrCreate()

In [2]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

BUCKET = "aws-ecommerce-data-abdelrahman"

# Historical data
RAW_PATH = f"s3://{BUCKET}/raw/ecommerce"

# API incremental data
API_RAW_PATH = f"s3://{BUCKET}/api_raw/orders"

# Processed Parquet data for Athena
PROCESSED_PATH = f"s3://{BUCKET}/processed/ecommerce"

In [1]:
# # ============================================================
# # REDSHIFT CONFIGURATION
# # ============================================================

# REDSHIFT_CONNECTION = "ecommerce-redshift-connection"

# REDSHIFT_DATABASE = "dev"

# REDSHIFT_SCHEMA = "public"

# # S3 temporary directory used by Spark/Glue
# REDSHIFT_TEMP_DIR = (
#     f"s3://{BUCKET}/redshift-temp/"
# )

In [3]:
# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def clean_column_names(df):

    for column in df.columns:

        new_name = (
            column.strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
        )

        df = df.withColumnRenamed(
            column,
            new_name
        )

    return df


def read_csv(filename):

    path = f"{RAW_PATH}/{filename}"

    print("Reading:", path)

    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)
        .csv(path)
    )


def write_parquet(df, table_name):

    output_path = (
        f"{PROCESSED_PATH}/{table_name}"
    )

    print(
        f"Writing {table_name} -> {output_path}"
    )

    (
        df.write
        .mode("overwrite")
        .format("parquet")
        .option("compression", "snappy")
        .save(output_path)
    )

In [5]:
# ============================================================
# 4. READ HISTORICAL DATA
# ============================================================

print("============================================")
print("READING HISTORICAL DATA")
print("============================================")


customers = read_csv("customers.csv")

categories = read_csv("categories.csv")

products = read_csv("products.csv")

departments = read_csv("departments.csv")

employees = read_csv("employees.csv")

suppliers = read_csv("suppliers.csv")

orders = read_csv("orders.csv")

order_details = read_csv("order_details.csv")

payments = read_csv("payments.csv")

product_suppliers = read_csv(
    "product_suppliers.csv"
)

shippers = read_csv("shippers.csv")

shipments = read_csv("shipments.csv")

READING HISTORICAL DATA
Reading: s3://aws-ecommerce-data-abdelrahman/raw/ecommerce/customers.csv
Reading: s3://aws-ecommerce-data-abdelrahman/raw/ecommerce/categories.csv
Reading: s3://aws-ecommerce-data-abdelrahman/raw/ecommerce/products.csv
Reading: s3://aws-ecommerce-data-abdelrahman/raw/ecommerce/departments.csv
Reading: s3://aws-ecommerce-data-abdelrahman/raw/ecommerce/employees.csv
Reading: s3://aws-ecommerce-data-abdelrahman/raw/ecommerce/suppliers.csv
Reading: s3://aws-ecommerce-data-abdelrahman/raw/ecommerce/orders.csv
Reading: s3://aws-ecommerce-data-abdelrahman/raw/ecommerce/order_details.csv
Reading: s3://aws-ecommerce-data-abdelrahman/raw/ecommerce/payments.csv
Reading: s3://aws-ecommerce-data-abdelrahman/raw/ecommerce/product_suppliers.csv
Reading: s3://aws-ecommerce-data-abdelrahman/raw/ecommerce/shippers.csv
Reading: s3://aws-ecommerce-data-abdelrahman/raw/ecommerce/shipments.csv


In [6]:
customers.printSchema()                  
categories.printSchema()
products.printSchema()
departments.printSchema()
employees.printSchema()
suppliers.printSchema()
orders.printSchema()
order_details.printSchema()
payments.printSchema()
product_suppliers.printSchema()
shippers.printSchema()
shipments.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- RegistrationDate: timestamp (nullable = true)

root
 |-- CategoryID: integer (nullable = true)
 |-- CategoryName: string (nullable = true)

root
 |-- ProductID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Cost: double (nullable = true)
 |-- Stock: integer (nullable = true)

root
 |-- DepartmentID: integer (nullable = true)
 |-- DepartmentName: string (nullable = true)

root
 |-- EmployeeID: integer (nullable = true)
 |-- ManagerID: integer (nullable = true)
 |-- DepartmentID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Salary: double

In [7]:
# ============================================================
# 5. CLEAN COLUMN NAMES
# ============================================================

print("============================================")
print("CLEANING COLUMN NAMES")
print("============================================")


customers = clean_column_names(customers)

categories = clean_column_names(categories)

products = clean_column_names(products)

departments = clean_column_names(departments)

employees = clean_column_names(employees)

suppliers = clean_column_names(suppliers)

orders = clean_column_names(orders)

order_details = clean_column_names(order_details)

payments = clean_column_names(payments)

product_suppliers = clean_column_names(
    product_suppliers
)

shippers = clean_column_names(shippers)

shipments = clean_column_names(shipments)

CLEANING COLUMN NAMES


In [8]:
# ============================================================
# 6. READ API JSON
# ============================================================

print("============================================")
print("READING API JSON")
print("============================================")

api_json = (
    spark.read
    .option("multiLine", True)
    .json(API_RAW_PATH)
)

print("API JSON schema:")

api_json.printSchema()

READING API JSON
API JSON schema:
root
 |-- metadata: struct (nullable = true)
 |    |-- event_id: string (nullable = true)
 |    |-- event_timestamp: string (nullable = true)
 |    |-- event_type: string (nullable = true)
 |    |-- source: string (nullable = true)
 |-- order: struct (nullable = true)
 |    |-- CustomerID: long (nullable = true)
 |    |-- OrderDate: string (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |-- Status: string (nullable = true)
 |-- order_details: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- Discount: double (nullable = true)
 |    |    |-- OrderDetailID: long (nullable = true)
 |    |    |-- OrderID: long (nullable = true)
 |    |    |-- ProductID: long (nullable = true)
 |    |    |-- Quantity: long (nullable = true)
 |    |    |-- UnitPrice: double (nullable = true)
 |-- payment: struct (nullable = true)
 |    |-- Amount: double (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |

In [10]:
import boto3

s3 = boto3.client("s3")

response = s3.list_objects_v2(
    Bucket="aws-ecommerce-data-abdelrahman",
    Prefix="api_raw/orders/"
)

for obj in response.get("Contents", []):
    print(obj["Key"])

api_raw/orders/order_2000001.json
api_raw/orders/order_2000002.json
api_raw/orders/order_2000003.json
api_raw/orders/order_2000004.json
api_raw/orders/order_2000005.json
api_raw/orders/order_2000006.json
api_raw/orders/order_2000007.json
api_raw/orders/order_2000008.json
api_raw/orders/order_2000009.json
api_raw/orders/order_2000010.json
api_raw/orders/order_2000011.json
api_raw/orders/order_2000012.json
api_raw/orders/order_2000013.json
api_raw/orders/order_2000014.json
api_raw/orders/order_2000015.json
api_raw/orders/order_2000016.json
api_raw/orders/order_2000017.json
api_raw/orders/order_2000018.json
api_raw/orders/order_2000019.json
api_raw/orders/order_2000020.json


In [11]:
# ============================================================
# 6. READ API JSON
# ============================================================

print("============================================")
print("READING API JSON")
print("============================================")

api_json = (
    spark.read
    .option("multiLine", True)
    .json(API_RAW_PATH)
)

print("API JSON schema:")

api_json.printSchema()

READING API JSON
API JSON schema:
root
 |-- metadata: struct (nullable = true)
 |    |-- event_id: string (nullable = true)
 |    |-- event_timestamp: string (nullable = true)
 |    |-- event_type: string (nullable = true)
 |    |-- source: string (nullable = true)
 |-- order: struct (nullable = true)
 |    |-- CustomerID: long (nullable = true)
 |    |-- OrderDate: string (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |-- Status: string (nullable = true)
 |-- order_details: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- Discount: double (nullable = true)
 |    |    |-- OrderDetailID: long (nullable = true)
 |    |    |-- OrderID: long (nullable = true)
 |    |    |-- ProductID: long (nullable = true)
 |    |    |-- Quantity: long (nullable = true)
 |    |    |-- UnitPrice: double (nullable = true)
 |-- payment: struct (nullable = true)
 |    |-- Amount: double (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |

In [13]:
from pyspark.sql import functions as F

BUCKET = "aws-ecommerce-data-abdelrahman"

API_RAW_PATH = f"s3://{BUCKET}/api_raw/orders/"

api_orders = (
    spark.read
    .option("multiLine", True)
    .json(API_RAW_PATH)
)

print("API records:", api_orders.count())

api_orders.printSchema()

API records: 20
root
 |-- metadata: struct (nullable = true)
 |    |-- event_id: string (nullable = true)
 |    |-- event_timestamp: string (nullable = true)
 |    |-- event_type: string (nullable = true)
 |    |-- source: string (nullable = true)
 |-- order: struct (nullable = true)
 |    |-- CustomerID: long (nullable = true)
 |    |-- OrderDate: string (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |-- Status: string (nullable = true)
 |-- order_details: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- Discount: double (nullable = true)
 |    |    |-- OrderDetailID: long (nullable = true)
 |    |    |-- OrderID: long (nullable = true)
 |    |    |-- ProductID: long (nullable = true)
 |    |    |-- Quantity: long (nullable = true)
 |    |    |-- UnitPrice: double (nullable = true)
 |-- payment: struct (nullable = true)
 |    |-- Amount: double (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |-- PaymentDate: st

In [14]:
api_orders_df = api_orders.select(
    
    F.col("order.OrderID")
        .cast("long")
        .alias("orderid"),

    F.col("order.CustomerID")
        .cast("long")
        .alias("customerid"),

    F.to_date(
        F.col("order.OrderDate")
    ).alias("orderdate"),

    F.upper(
        F.trim(
            F.col("order.Status")
        )
    ).alias("status"),

    # API metadata
    F.col("metadata.event_id")
        .alias("event_id"),

    F.to_timestamp(
        F.col("metadata.event_timestamp")
    ).alias("event_timestamp"),

    F.col("metadata.event_type")
        .alias("event_type"),

    F.col("metadata.source")
        .alias("source")
)

api_orders_df.show(10, truncate=False)

+-------+----------+----------+---------+------------------------------------+--------------------------+-------------+------------+
|orderid|customerid|orderdate |status   |event_id                            |event_timestamp           |event_type   |source      |
+-------+----------+----------+---------+------------------------------------+--------------------------+-------------+------------+
|2000015|49526     |2026-09-22|CONFIRMED|451c8a50-3c19-487c-9384-7d162a615693|2026-09-22 15:22:19.112439|ORDER_CREATED|ordering_api|
|2000004|286647    |2026-09-22|CONFIRMED|a571ef27-1db7-4af4-a1cc-c86e87a2e2e8|2026-09-22 15:22:19.093936|ORDER_CREATED|ordering_api|
|2000009|20885     |2026-09-22|CONFIRMED|5189f1a5-752f-490e-a9d7-c1c7b8d37499|2026-09-22 15:22:19.104562|ORDER_CREATED|ordering_api|
|2000013|242192    |2026-09-22|CONFIRMED|aef69fdb-0f4f-4d91-bda6-b697358022a7|2026-09-22 15:22:19.110068|ORDER_CREATED|ordering_api|
|2000018|266897    |2026-09-22|CONFIRMED|9140f9c4-a649-4f39-91d2-4e5b

In [15]:
api_order_details_df = (
    api_orders
    .select(
        
        F.col("order.OrderID")
            .cast("long")
            .alias("orderid"),

        F.explode(
            F.col("order_details")
        ).alias("detail")
    )
)
api_order_details_df.show()

+-------+--------------------+
|orderid|              detail|
+-------+--------------------+
|2000015|{15.0, 5000034, 2...|
|2000015|{5.0, 5000035, 20...|
|2000015|{15.0, 5000036, 2...|
|2000015|{15.0, 5000037, 2...|
|2000004|{10.0, 5000007, 2...|
|2000004|{5.0, 5000008, 20...|
|2000004|{0.0, 5000009, 20...|
|2000004|{15.0, 5000010, 2...|
|2000009|{15.0, 5000018, 2...|
|2000009|{15.0, 5000019, 2...|
|2000009|{0.0, 5000020, 20...|
|2000009|{0.0, 5000021, 20...|
|2000013|{0.0, 5000028, 20...|
|2000013|{5.0, 5000029, 20...|
|2000013|{0.0, 5000030, 20...|
|2000013|{0.0, 5000031, 20...|
|2000018|{15.0, 5000042, 2...|
|2000018|{0.0, 5000043, 20...|
|2000018|{0.0, 5000044, 20...|
|2000001|{15.0, 5000001, 2...|
+-------+--------------------+
only showing top 20 rows


In [16]:
api_order_details_df = api_order_details_df.select(

    "orderid",

    F.col("detail.OrderDetailID")
        .cast("long")
        .alias("orderdetailid"),

    F.col("detail.ProductID")
        .cast("long")
        .alias("productid"),

    F.col("detail.Quantity")
        .cast("long")
        .alias("quantity"),

    F.col("detail.UnitPrice")
        .cast("decimal(12,2)")
        .alias("unitprice"),

    F.col("detail.Discount")
        .cast("decimal(12,2)")
        .alias("discount")
)
api_order_details_df.show()

+-------+-------------+---------+--------+---------+--------+
|orderid|orderdetailid|productid|quantity|unitprice|discount|
+-------+-------------+---------+--------+---------+--------+
|2000015|      5000034|     1744|       2|   852.25|   15.00|
|2000015|      5000035|     1837|       3|   389.21|    5.00|
|2000015|      5000036|     8244|       5|   460.40|   15.00|
|2000015|      5000037|     7153|       1|   591.57|   15.00|
|2000004|      5000007|     7197|       2|   594.21|   10.00|
|2000004|      5000008|     7824|       5|   529.92|    5.00|
|2000004|      5000009|     4306|       2|   596.61|    0.00|
|2000004|      5000010|      723|       2|   161.34|   15.00|
|2000009|      5000018|      761|       5|   874.55|   15.00|
|2000009|      5000019|     5836|       4|   105.24|   15.00|
|2000009|      5000020|     4003|       3|   533.03|    0.00|
|2000009|      5000021|       93|       1|   129.79|    0.00|
|2000013|      5000028|      405|       4|   789.95|    0.00|
|2000013

In [17]:
api_payments_df = api_orders.select(

    F.col("payment.PaymentID")
        .cast("long")
        .alias("paymentid"),

    F.col("payment.OrderID")
        .cast("long")
        .alias("orderid"),

    F.trim(
        F.col("payment.PaymentMethod")
    ).alias("paymentmethod"),

    F.to_date(
        F.col("payment.PaymentDate")
    ).alias("paymentdate"),

    F.col("payment.Amount")
        .cast("decimal(12,2)")
        .alias("amount"),

    F.upper(
        F.trim(
            F.col("payment.PaymentStatus")
        )
    ).alias("paymentstatus")
)

api_payments_df.show(10)

+---------+-------+----------------+-----------+-------+-------------+
|paymentid|orderid|   paymentmethod|paymentdate| amount|paymentstatus|
+---------+-------+----------------+-----------+-------+-------------+
|  2000015|2000015|Cash on Delivery| 2026-09-22|5770.08|    COMPLETED|
|  2000004|2000004|Cash on Delivery| 2026-09-22|5811.79|    COMPLETED|
|  2000009|2000009|     Credit Card| 2026-09-22|6666.04|    COMPLETED|
|  2000013|2000013|   Bank Transfer| 2026-09-22|6723.20|    COMPLETED|
|  2000018|2000018|     Credit Card| 2026-09-22|1149.67|    COMPLETED|
|  2000001|2000001|          PayPal| 2026-09-22|4631.64|    COMPLETED|
|  2000007|2000007|          PayPal| 2026-09-22|5693.25|    COMPLETED|
|  2000012|2000012|Cash on Delivery| 2026-09-22|2541.79|    COMPLETED|
|  2000016|2000016|Cash on Delivery| 2026-09-22|2751.10|    COMPLETED|
|  2000011|2000011|   Bank Transfer| 2026-09-22|7273.63|    COMPLETED|
+---------+-------+----------------+-----------+-------+-------------+
only s

In [18]:
api_shipments_df = api_orders.select(

    F.col("shipment.ShipmentID")
        .cast("long")
        .alias("shipmentid"),

    F.col("shipment.OrderID")
        .cast("long")
        .alias("orderid"),

    F.col("shipment.ShipperID")
        .cast("long")
        .alias("shipperid"),

    F.to_date(
        F.col("shipment.ShipDate")
    ).alias("shipdate"),

    F.to_date(
        F.col("shipment.DeliveryDate")
    ).alias("deliverydate")
)

api_shipments_df.show(10)

+----------+-------+---------+----------+------------+
|shipmentid|orderid|shipperid|  shipdate|deliverydate|
+----------+-------+---------+----------+------------+
|    500015|2000015|      707|2026-09-23|  2026-09-28|
|    500004|2000004|      883|2026-09-23|  2026-09-27|
|    500009|2000009|      455|2026-09-23|  2026-09-28|
|    500013|2000013|      102|2026-09-23|  2026-09-26|
|    500018|2000018|      630|2026-09-23|  2026-09-26|
|    500001|2000001|      355|2026-09-23|  2026-09-27|
|    500007|2000007|      678|2026-09-23|  2026-09-26|
|    500012|2000012|      262|2026-09-23|  2026-09-25|
|    500016|2000016|      548|2026-09-23|  2026-09-28|
|    500011|2000011|      977|2026-09-23|  2026-09-26|
+----------+-------+---------+----------+------------+
only showing top 10 rows


In [19]:
api_order_summary_df = api_orders.select(

    F.col("order.OrderID")
        .cast("long")
        .alias("orderid"),

    F.col("summary.subtotal")
        .cast("decimal(12,2)")
        .alias("subtotal"),

    F.col("summary.discount")
        .cast("decimal(12,2)")
        .alias("discount"),

    F.col("summary.tax_rate")
        .cast("decimal(5,2)")
        .alias("tax_rate"),

    F.col("summary.tax_amount")
        .cast("decimal(12,2)")
        .alias("tax_amount"),

    F.col("summary.shipping_cost")
        .cast("decimal(12,2)")
        .alias("shipping_cost"),

    F.col("summary.total_amount")
        .cast("decimal(12,2)")
        .alias("total_amount")
)
api_order_summary_df.show(10)

+-------+--------+--------+--------+----------+-------------+------------+
|orderid|subtotal|discount|tax_rate|tax_amount|shipping_cost|total_amount|
+-------+--------+--------+--------+----------+-------------+------------+
|2000015| 5017.61|  748.09|    0.14|    702.47|        50.00|     5770.08|
|2000004| 5054.20|  299.72|    0.14|    707.59|        50.00|     5811.79|
|2000009| 5803.54|  719.05|    0.14|    812.50|        50.00|     6666.04|
|2000013| 5853.68|   46.46|    0.14|    819.52|        50.00|     6723.20|
|2000018|  964.62|   30.11|    0.14|    135.05|        50.00|     1149.67|
|2000001| 4018.98|  668.84|    0.14|    562.66|        50.00|     4631.64|
|2000007| 4950.22|  379.91|    0.14|    693.03|        50.00|     5693.25|
|2000012| 2185.78|  106.03|    0.14|    306.01|        50.00|     2541.79|
|2000016| 2369.39|  391.93|    0.14|    331.71|        50.00|     2751.10|
|2000011| 6336.52|  695.53|    0.14|    887.11|        50.00|     7273.63|
+-------+--------+-------

In [20]:
historical_orders = orders.select(
    "orderid",
    "customerid",
    "orderdate",
    "status"
)

In [21]:
new_orders = api_orders_df.select(
    "orderid",
    "customerid",
    "orderdate",
    "status"
)

In [22]:
all_orders = (
    historical_orders
    .unionByName(new_orders)
    .dropDuplicates(["orderid"])
)
all_orders.show(10)

+-------+----------+-------------------+----------+
|orderid|customerid|          orderdate|    status|
+-------+----------+-------------------+----------+
|      3|      8030|2026-01-14 00:00:00| Delivered|
|      9|      3854|2025-01-16 00:00:00| Delivered|
|     11|      7696|2026-02-08 00:00:00|   Pending|
|     14|      3280|2024-03-04 00:00:00| Delivered|
|     17|      8043|2024-04-24 00:00:00| Cancelled|
|     28|      2916|2025-03-09 00:00:00|Processing|
|     29|       873|2024-05-19 00:00:00| Cancelled|
|     30|      4535|2026-03-05 00:00:00| Cancelled|
|     32|      8261|2024-05-06 00:00:00|   Pending|
|     33|      3053|2025-07-25 00:00:00|Processing|
+-------+----------+-------------------+----------+
only showing top 10 rows


In [23]:
print("Historical orders:", historical_orders.count())
print("API orders:", new_orders.count())
print("Combined orders:", all_orders.count())

Historical orders: 50000
API orders: 20
Combined orders: 50020


In [24]:
historical_orders_details = order_details.select(
    "OrderDetailID",
    "OrderID",
    "ProductID",
    "Quantity",
    "UnitPrice",
    "Discount"
)

In [25]:
new_orders_details = api_order_details_df.select(
    "OrderDetailID",
    "OrderID",
    "ProductID",
    "Quantity",
    "UnitPrice",
    "Discount"
)

In [26]:
all_orders_details = (
    historical_orders_details
    .unionByName(new_orders_details)
    .dropDuplicates(["OrderDetailID"])
)
all_orders_details.show(10)

+-------------+-------+---------+--------+---------+--------+
|OrderDetailID|OrderID|ProductID|Quantity|UnitPrice|Discount|
+-------------+-------+---------+--------+---------+--------+
|           28|  21929|      412|       8|  1768.21|    0.00|
|           29|  27162|      707|       3|   521.53|    0.00|
|           30|  44774|      432|       6|   876.51|    0.00|
|           33|  44871|      558|       6|   1989.3|   15.00|
|           93|  46841|      969|       7|  2243.46|    5.00|
|          151|  31224|      232|       3|  2916.72|   10.00|
|          171|  49101|      899|       9|   1128.4|    5.00|
|          200|  25519|      335|       9|  2999.57|   10.00|
|          208|  42394|      127|       6|   2271.6|    5.00|
|          221|  18333|      752|       4|  1950.81|   10.00|
+-------------+-------+---------+--------+---------+--------+
only showing top 10 rows


In [27]:
print("Historical orders_details:", historical_orders_details.count())
print("API orders_details:", new_orders_details.count())
print("Combined orders_details:", all_orders_details.count())

Historical orders_details: 100000
API orders_details: 48
Combined orders_details: 100048


In [28]:
historical_payments = payments.select(
    "paymentid",
    "orderid",
    "paymentmethod",
    "paymentdate",
    "amount"
)

In [29]:
new_orders_payments = api_payments_df.select(
    "paymentid",
    "orderid",
    "paymentmethod",
    "paymentdate",
    "amount"
)

In [30]:
all_payments = (
    historical_payments
    .unionByName(new_orders_payments)
    .dropDuplicates(["paymentid"])
)
all_payments.show(10)

+---------+-------+-------------+-------------------+-------+
|paymentid|orderid|paymentmethod|        paymentdate| amount|
+---------+-------+-------------+-------------------+-------+
|        3|  10641|  Credit Card|2025-08-05 00:00:00| 609.92|
|        9|  20619|         Cash|2024-12-06 00:00:00|3859.07|
|       11|  35348|       PayPal|2024-09-24 00:00:00|1837.06|
|       14|   2807|         Cash|2024-01-01 00:00:00| 1784.9|
|       17|  16704|Bank Transfer|2026-06-30 00:00:00|4311.38|
|       28|  31938|         Cash|2025-01-28 00:00:00| 4893.5|
|       29|  48409|         Cash|2026-03-15 00:00:00|3070.39|
|       30|  37096|  Credit Card|2024-12-24 00:00:00| 3453.1|
|       32|  32979|   Debit Card|2025-11-10 00:00:00|4464.56|
|       33|  40215|       PayPal|2025-10-03 00:00:00| 851.07|
+---------+-------+-------------+-------------------+-------+
only showing top 10 rows


In [31]:
print("Historical payments:", historical_payments.count())
print("API payments:", new_orders_payments.count())
print("Combined payments:", all_payments.count())

Historical payments: 45000
API payments: 20
Combined payments: 45020


In [32]:
historical_shipments = shipments.select(
   "ShipmentID",
    "OrderID",
    "ShipperID",
    "ShipDate",
    "DeliveryDate"
)

In [33]:
new_orders_shipments = api_shipments_df.select(
   "ShipmentID",
    "OrderID",
    "ShipperID",
    "ShipDate",
    "DeliveryDate"
)

In [34]:
all_shipments = (
    historical_shipments
    .unionByName(new_orders_shipments)
    .dropDuplicates(["ShipmentID"])
)
all_shipments.show(10)

+----------+-------+---------+-------------------+-------------------+
|ShipmentID|OrderID|ShipperID|           ShipDate|       DeliveryDate|
+----------+-------+---------+-------------------+-------------------+
|        28|  43728|        8|2025-03-08 00:00:00|2025-03-15 00:00:00|
|        29|  15494|        2|2025-10-29 00:00:00|2025-10-31 00:00:00|
|        30|   7088|        3|2026-09-01 00:00:00|2026-09-07 00:00:00|
|        33|  33162|        1|2024-05-28 00:00:00|2024-06-03 00:00:00|
|        93|  15808|        7|2024-06-04 00:00:00|2024-06-07 00:00:00|
|       151|  19260|        3|2024-07-28 00:00:00|2024-08-06 00:00:00|
|       171|  16934|        7|2024-08-02 00:00:00|2024-08-04 00:00:00|
|       200|  22106|        3|2024-11-05 00:00:00|2024-11-09 00:00:00|
|       208|  31687|        1|2024-08-11 00:00:00|2024-08-20 00:00:00|
|       221|  13122|        7|2024-12-19 00:00:00|2024-12-23 00:00:00|
+----------+-------+---------+-------------------+-------------------+
only s

In [35]:
print("Historical shipments:", historical_shipments.count())
print("API shipments:", new_orders_shipments.count())
print("Combined shipments:", all_shipments.count())

Historical shipments: 40000
API shipments: 20
Combined shipments: 40020


In [36]:
all_shipments.groupBy("ShipmentID") \
    .count() \
    .filter("count > 1") \
    .show()

+----------+-----+
|ShipmentID|count|
+----------+-----+
+----------+-----+


In [37]:
all_shipments.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in all_shipments.columns
]).show()

+----------+-------+---------+--------+------------+
|ShipmentID|OrderID|ShipperID|ShipDate|DeliveryDate|
+----------+-------+---------+--------+------------+
|         0|      0|        0|       0|           0|
+----------+-------+---------+--------+------------+


In [38]:
all_shipments = all_shipments.dropna(how="all")

In [39]:
print("Historical shipments:", historical_shipments.count())
print("API shipments:", new_orders_shipments.count())
print("Combined shipments:", all_shipments.count())

Historical shipments: 40000
API shipments: 20
Combined shipments: 40020


In [40]:
all_orders_details = (
    all_orders_details

    # Make sure types are consistent
    .withColumn("orderdetailid", F.col("orderdetailid").cast("long"))
    .withColumn("orderid", F.col("orderid").cast("long"))
    .withColumn("productid", F.col("productid").cast("long"))
    .withColumn("quantity", F.col("quantity").cast("long"))
    .withColumn("unitprice", F.col("unitprice").cast("decimal(12,2)"))
    .withColumn("discount", F.col("discount").cast("decimal(12,2)"))

    # Handle negative values
    .withColumn(
        "quantity",
        F.when(F.col("quantity") < 0, 0)
         .otherwise(F.col("quantity"))
    )

    .withColumn(
        "unitprice",
        F.when(F.col("unitprice") < 0, 0)
         .otherwise(F.col("unitprice"))
    )

    .withColumn(
        "discount",
        F.when(F.col("discount") < 0, 0)
         .otherwise(F.col("discount"))
    )

    # Recalculate amounts
    .withColumn(
        "gross_amount",
        F.col("quantity") * F.col("unitprice")
    )

    .withColumn(
        "discount_amount",
        F.col("gross_amount") * F.col("discount") / 100
    )

    .withColumn(
        "total_amount",
        F.col("gross_amount") - F.col("discount_amount")
    )
)

In [41]:
all_orders_details.show(2)

+-------------+-------+---------+--------+---------+--------+------------+---------------+------------+
|orderdetailid|orderid|productid|quantity|unitprice|discount|gross_amount|discount_amount|total_amount|
+-------------+-------+---------+--------+---------+--------+------------+---------------+------------+
|           28|  21929|      412|       8|  1768.21|    0.00|    14145.68|       0.000000|14145.680000|
|           29|  27162|      707|       3|   521.53|    0.00|     1564.59|       0.000000| 1564.590000|
+-------------+-------+---------+--------+---------+--------+------------+---------------+------------+
only showing top 2 rows


In [42]:
all_orders_details.count()

100048


In [43]:
all_api_orders = (
    api_orders_df
    .withColumn("year", F.year("orderdate"))
    .withColumn("month", F.month("orderdate"))
)
all_api_orders.show(10)

+-------+----------+----------+---------+--------------------+--------------------+-------------+------------+----+-----+
|orderid|customerid| orderdate|   status|            event_id|     event_timestamp|   event_type|      source|year|month|
+-------+----------+----------+---------+--------------------+--------------------+-------------+------------+----+-----+
|2000015|     49526|2026-09-22|CONFIRMED|451c8a50-3c19-487...|2026-09-22 15:22:...|ORDER_CREATED|ordering_api|2026|    9|
|2000004|    286647|2026-09-22|CONFIRMED|a571ef27-1db7-4af...|2026-09-22 15:22:...|ORDER_CREATED|ordering_api|2026|    9|
|2000009|     20885|2026-09-22|CONFIRMED|5189f1a5-752f-490...|2026-09-22 15:22:...|ORDER_CREATED|ordering_api|2026|    9|
|2000013|    242192|2026-09-22|CONFIRMED|aef69fdb-0f4f-4d9...|2026-09-22 15:22:...|ORDER_CREATED|ordering_api|2026|    9|
|2000018|    266897|2026-09-22|CONFIRMED|9140f9c4-a649-4f3...|2026-09-22 15:22:...|ORDER_CREATED|ordering_api|2026|    9|
|2000001|    366336|2026

In [44]:
API_CURATED_PATH = (
    f"s3://{BUCKET}/processed/ecommerce/api_orders"
)

(
    all_api_orders
    .write
    .mode("append")
    .format("parquet")
    .partitionBy("year", "month")
    .option("compression", "snappy")
    .save(API_CURATED_PATH)
)

In [45]:
dim_customer = (
    customers
    .select(
        F.col("customerid").cast("long").alias("customer_id"),
        F.trim("firstname").alias("first_name"),
        F.trim("lastname").alias("last_name"),
        F.concat_ws(
            " ",
            F.trim("firstname"),
            F.trim("lastname")
        ).alias("full_name"),
        F.lower(F.trim("email")).alias("email"),
        F.trim("city").alias("city"),
        F.trim("country").alias("country"),
        F.to_date("registrationdate").alias("registration_date")
    )
    .dropDuplicates(["customer_id"])
)

In [46]:
dim_category = (
    categories
    .select(
        F.col("categoryid").cast("long").alias("category_id"),
        F.trim("categoryname").alias("category_name")
    )
    .dropDuplicates(["category_id"])
)

In [47]:
dim_department = (
    departments
    .select(
        F.col("departmentid").cast("long").alias("department_id"),
        F.trim("departmentname").alias("department_name")
    )
    .dropDuplicates(["department_id"])
)

In [48]:
dim_product = (
    products
    .select(
        F.col("productid").cast("long").alias("product_id"),
        F.col("categoryid").cast("long").alias("category_id"),
        F.trim("productname").alias("product_name"),
        F.trim("brand").alias("brand"),
        F.col("price").cast("decimal(12,2)").alias("price"),
        F.col("cost").cast("decimal(12,2)").alias("cost"),
        F.col("stock").cast("int").alias("stock")
    )
    .dropDuplicates(["product_id"])
)

In [49]:
dim_supplier = (
    suppliers
    .select(
        F.col("supplierid").cast("long").alias("supplier_id"),
        F.trim("suppliername").alias("supplier_name"),
        F.trim("country").alias("country")
    )
    .dropDuplicates(["supplier_id"])
)

In [50]:
dim_employee = (
    employees
    .select(
        F.col("employeeid").cast("long").alias("employee_id"),
        F.col("managerid").cast("long").alias("manager_id"),
        F.col("departmentid").cast("long").alias("department_id"),
        F.trim("firstname").alias("first_name"),
        F.trim("lastname").alias("last_name"),
        F.concat_ws(
            " ",
            F.trim("firstname"),
            F.trim("lastname")
        ).alias("full_name"),
        F.col("salary").cast("decimal(12,2)").alias("salary"),
        F.to_date("hiredate").alias("hire_date")
    )
    .dropDuplicates(["employee_id"])
)

In [51]:
dim_shipper = (
    shippers
    .select(
        F.col("shipperid").cast("long").alias("shipper_id"),
        F.trim("companyname").alias("company_name")
    )
    .dropDuplicates(["shipper_id"])
)

In [52]:
date_range = (
    all_orders
    .select(
        F.to_date("orderdate").alias("date")
    )
    .filter(F.col("date").isNotNull())
    .agg(
        F.min("date").alias("min_date"),
        F.max("date").alias("max_date")
    )
)

In [53]:
dim_date = (
    date_range
    .select(
        F.explode(
            F.sequence(
                F.col("min_date"),
                F.col("max_date"),
                F.expr("interval 1 day")
            )
        ).alias("date")
    )
)

In [54]:
dim_date = (
    dim_date
    .withColumn(
        "date_id",
        F.date_format("date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn(
        "month_name",
        F.date_format("date", "MMMM")
    )
    .withColumn(
        "day",
        F.dayofmonth("date")
    )
    .withColumn(
        "day_of_week",
        F.dayofweek("date")
    )
    .withColumn(
        "day_name",
        F.date_format("date", "EEEE")
    )
    .withColumn(
        "week_of_year",
        F.weekofyear("date")
    )
)

In [55]:
payment_methods = (
    all_payments
    .select(
        F.upper(
            F.trim("paymentmethod")
        ).alias("payment_method")
    )
    .filter(
        F.col("payment_method").isNotNull()
    )
    .distinct()
)

In [56]:
window_payment = Window.orderBy("payment_method")

dim_payment_method = (
    payment_methods
    .withColumn(
        "payment_method_id",
        F.row_number().over(window_payment)
    )
    .select(
        "payment_method_id",
        "payment_method"
    )
)

In [57]:
order_statuses = (
    all_orders
    .select(
        F.upper(
            F.trim("status")
        ).alias("order_status")
    )
    .filter(
        F.col("order_status").isNotNull()
    )
    .distinct()
)

In [58]:
window_status = Window.orderBy("order_status")

dim_order_status = (
    order_statuses
    .withColumn(
        "order_status_id",
        F.row_number().over(window_status)
    )
    .select(
        "order_status_id",
        "order_status"
    )
)

In [59]:
fact_order = (
    all_orders
    .select(
        F.col("orderid").cast("long").alias("order_id"),
        F.col("customerid").cast("long").alias("customer_id"),
        F.to_date("orderdate").alias("order_date"),
        F.date_format(
            F.to_date("orderdate"),
            "yyyyMMdd"
        ).cast("int").alias("date_id"),
        F.upper(
            F.trim("status")
        ).alias("order_status")
    )
)

In [60]:
fact_order = (
    fact_order
    .join(
        dim_order_status,
        fact_order.order_status ==
        dim_order_status.order_status,
        "left"
    )
    .select(
        "order_id",
        "customer_id",
        "date_id",
        "order_date",
        "order_status_id"
    )
)

In [61]:
fact_order_detail = (
    all_orders_details
    .select(
        F.col("orderdetailid")
            .cast("long")
            .alias("order_detail_id"),

        F.col("orderid")
            .cast("long")
            .alias("order_id"),

        F.col("productid")
            .cast("long")
            .alias("product_id"),

        F.col("quantity")
            .cast("int")
            .alias("quantity"),

        F.col("unitprice")
            .cast("decimal(12,2)")
            .alias("unit_price"),

        F.col("discount")
            .cast("decimal(12,2)")
            .alias("discount"),

        F.col("gross_amount")
            .cast("decimal(14,2)")
            .alias("gross_amount"),

        F.col("discount_amount")
            .cast("decimal(14,2)")
            .alias("discount_amount"),

        F.col("total_amount")
            .cast("decimal(14,2)")
            .alias("total_amount")
    )
    .dropDuplicates(["order_detail_id"])
)

In [156]:
fact_order_detail.show(5)

+---------------+--------+----------+--------+----------+--------+------------+---------------+------------+

|order_detail_id|order_id|product_id|quantity|unit_price|discount|gross_amount|discount_amount|total_amount|

+---------------+--------+----------+--------+----------+--------+------------+---------------+------------+

|             28| 1135417|      7156|       3|   2423.09|    5.00|     7269.27|         363.46|     6905.81|

|             29| 1941313|      8432|       8|   1981.19|    0.00|    15849.52|           0.00|    15849.52|

|             30| 1711404|      2125|      10|    421.11|    0.00|     4211.10|           0.00|     4211.10|

|             33|  454099|      2748|       2|   1000.65|   10.00|     2001.30|         200.13|     1801.17|

|             93|   67034|      8522|       1|   1666.62|    5.00|     1666.62|          83.33|     1583.29|

+---------------+--------+----------+--------+----------+--------+------------+---------------+------------+

only showi

In [62]:
fact_payment_base = (
    all_payments
    .select(
        F.col("paymentid")
            .cast("long")
            .alias("payment_id"),

        F.col("orderid")
            .cast("long")
            .alias("order_id"),

        F.to_date("paymentdate")
            .alias("payment_date"),

        F.date_format(
            F.to_date("paymentdate"),
            "yyyyMMdd"
        )
        .cast("int")
        .alias("date_id"),

        F.upper(
            F.trim("paymentmethod")
        )
        .alias("payment_method"),

        F.col("amount")
            .cast("decimal(14,2)")
            .alias("amount")
    )
)

In [63]:
fact_payment = (
    fact_payment_base
    .join(
        dim_payment_method,
        fact_payment_base.payment_method ==
        dim_payment_method.payment_method,
        "left"
    )
    .select(
        "payment_id",
        "order_id",
        "date_id",
        "payment_date",
        "payment_method_id",
        "amount"
    )
)

In [64]:
fact_shipment = (
    all_shipments
    .select(
        F.col("shipmentid")
            .cast("long")
            .alias("shipment_id"),

        F.col("orderid")
            .cast("long")
            .alias("order_id"),

        F.col("shipperid")
            .cast("long")
            .alias("shipper_id"),

        F.to_date("shipdate")
            .alias("ship_date"),

        F.to_date("deliverydate")
            .alias("delivery_date")
    )
    .withColumn(
        "shipping_days",
        F.datediff(
            F.col("delivery_date"),
            F.col("ship_date")
        )
    )
)

In [65]:
fact_shipment.show(2)

+-----------+--------+----------+----------+-------------+-------------+
|shipment_id|order_id|shipper_id| ship_date|delivery_date|shipping_days|
+-----------+--------+----------+----------+-------------+-------------+
|         28|   43728|         8|2025-03-08|   2025-03-15|            7|
|         29|   15494|         2|2025-10-29|   2025-10-31|            2|
+-----------+--------+----------+----------+-------------+-------------+
only showing top 2 rows


In [66]:
customer_sales_base = (
    fact_order_detail
    .join(
        fact_order.select(
            "order_id",
            "customer_id",
            "order_date"
        ),
        "order_id",
        "inner"
    )
)

In [67]:
fact_customer_sales = (
    customer_sales_base
    .groupBy("customer_id")
    .agg(
        F.countDistinct(
            "order_id"
        ).alias("total_orders"),

        F.sum(
            "quantity"
        ).alias("total_quantity"),

        F.sum(
            "gross_amount"
        ).alias("gross_sales"),

        F.sum(
            "discount_amount"
        ).alias("total_discount"),

        F.sum(
            "total_amount"
        ).alias("total_sales"),

        F.avg(
            "total_amount"
        ).alias("average_order_line"),

        F.min(
            "order_date"
        ).alias("first_order_date"),

        F.max(
            "order_date"
        ).alias("last_order_date")
    )
)

In [68]:
fact_customer_sales = (
    fact_customer_sales
    .withColumn(
        "customer_segment",
        F.when(
            F.col("total_sales") >= 10000,
            "VIP"
        )
        .when(
            F.col("total_sales") >= 5000,
            "PREMIUM"
        )
        .when(
            F.col("total_sales") >= 1000,
            "REGULAR"
        )
        .otherwise(
            "LOW_VALUE"
        )
    )
)

In [69]:
fact_customer_sales.show(1)

+-----------+------------+--------------+-----------+--------------+-----------+------------------+----------------+---------------+----------------+
|customer_id|total_orders|total_quantity|gross_sales|total_discount|total_sales|average_order_line|first_order_date|last_order_date|customer_segment|
+-----------+------------+--------------+-----------+--------------+-----------+------------------+----------------+---------------+----------------+
|       1027|           3|            31|   23451.92|        108.60|   23343.32|       4668.664000|      2024-12-02|     2025-11-27|             VIP|
+-----------+------------+--------------+-----------+--------------+-----------+------------------+----------------+---------------+----------------+
only showing top 1 row


In [70]:
fact_product_sales = (
    fact_order_detail
    .groupBy("product_id")
    .agg(
        F.countDistinct(
            "order_id"
        ).alias("total_orders"),

        F.sum(
            "quantity"
        ).alias("units_sold"),

        F.sum(
            "gross_amount"
        ).alias("gross_sales"),

        F.sum(
            "discount_amount"
        ).alias("total_discount"),

        F.sum(
            "total_amount"
        ).alias("total_sales"),

        F.avg(
            "unit_price"
        ).alias("average_unit_price")
    )
)

In [71]:
bridge_product_supplier = (
    product_suppliers
    .select(
        F.col("productid")
            .cast("long")
            .alias("product_id"),

        F.col("supplierid")
            .cast("long")
            .alias("supplier_id")
    )
    .dropDuplicates(
        ["product_id", "supplier_id"]
    )
)

In [72]:
write_parquet(dim_customer, "dim_customer")
write_parquet(dim_product, "dim_product")
write_parquet(dim_category, "dim_category")
write_parquet(dim_department, "dim_department")
write_parquet(dim_supplier, "dim_supplier")
write_parquet(dim_employee, "dim_employee")
write_parquet(dim_shipper, "dim_shipper")
write_parquet(dim_date, "dim_date")
write_parquet(dim_payment_method, "dim_payment_method")
write_parquet(dim_order_status, "dim_order_status")

Writing dim_customer -> s3://aws-ecommerce-data-abdelrahman/processed/ecommerce/dim_customer
Writing dim_product -> s3://aws-ecommerce-data-abdelrahman/processed/ecommerce/dim_product
Writing dim_category -> s3://aws-ecommerce-data-abdelrahman/processed/ecommerce/dim_category
Writing dim_department -> s3://aws-ecommerce-data-abdelrahman/processed/ecommerce/dim_department
Writing dim_supplier -> s3://aws-ecommerce-data-abdelrahman/processed/ecommerce/dim_supplier
Writing dim_employee -> s3://aws-ecommerce-data-abdelrahman/processed/ecommerce/dim_employee
Writing dim_shipper -> s3://aws-ecommerce-data-abdelrahman/processed/ecommerce/dim_shipper
Writing dim_date -> s3://aws-ecommerce-data-abdelrahman/processed/ecommerce/dim_date
Writing dim_payment_method -> s3://aws-ecommerce-data-abdelrahman/processed/ecommerce/dim_payment_method
Writing dim_order_status -> s3://aws-ecommerce-data-abdelrahman/processed/ecommerce/dim_order_status


In [73]:
write_parquet(fact_order, "fact_order")
write_parquet(fact_order_detail, "fact_order_detail")
write_parquet(fact_payment, "fact_payment")
write_parquet(fact_shipment, "fact_shipment")
write_parquet(fact_customer_sales, "fact_customer_sales")
write_parquet(fact_product_sales, "fact_product_sales")

Writing fact_order -> s3://aws-ecommerce-data-abdelrahman/processed/ecommerce/fact_order
Writing fact_order_detail -> s3://aws-ecommerce-data-abdelrahman/processed/ecommerce/fact_order_detail
Writing fact_payment -> s3://aws-ecommerce-data-abdelrahman/processed/ecommerce/fact_payment
Writing fact_shipment -> s3://aws-ecommerce-data-abdelrahman/processed/ecommerce/fact_shipment
Writing fact_customer_sales -> s3://aws-ecommerce-data-abdelrahman/processed/ecommerce/fact_customer_sales
Writing fact_product_sales -> s3://aws-ecommerce-data-abdelrahman/processed/ecommerce/fact_product_sales


In [74]:
write_parquet(bridge_product_supplier,"bridge_product_supplier")

Writing bridge_product_supplier -> s3://aws-ecommerce-data-abdelrahman/processed/ecommerce/bridge_product_supplier
